## SRP489847

**paper:** [10.1016/j.scitotenv.2024.178262](https://www.sciencedirect.com/science/article/pii/S0048969724084201) - A neonicotinoid pesticide causes tissue-specific gene expression changes in bumble bees, 2025
* [PMID: 40069737](https://pmc.ncbi.nlm.nih.gov/articles/PMC11900027/) - Acute and chronic pesticide exposure trigger fundamentally different molecular responses in bumble bee brains, 2025

**date, curator:** 2024-08-27, Sara Carsanaro

**notes**
* i removed 70 of the 96 libraries because they were all treated with pesticides
* updated infoStage to adult for all samples because we know there are 2 weeks old + treatment time, but treatment time isn't specified for libraries so i think adult is more appropriate. this is also the dev stage they list
* i listed all strains as audax although it's not really a strain it's a subspecies
* library selection in SRA is polyA
* per methods library prep is NEBNext Ultra II Directional RNA Library Prep Kit
* some libraries are described in one paper, some are described in the other paper

### annotation summary
run this after annotation is complete

In [34]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,FEMUR,UBERON:6007150,insect segment of leg,missing child term
1,TUBULES,UBERON:0001054,Malpighian tubule,perfect match
16,BRAIN,UBERON:0000955,brain,perfect match


In [35]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,ADULT,UBERON:0000113,post-juvenile adult stage,perfect match
16,adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP489847"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_25952/3311601719.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
Be patient, it may take a few minutes.
 93%|███████████████████████████████████████▊   | 89/96 [01:34<00:07,  1.00s/it]curl: (22) 

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,,,,,,FEMUR,ADULT,,,,F,audax,,30195,,,,,,FCONT.9_FEMUR,SAMN43369244,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-9-LEG_S52,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403796,C2BC35F34CB9DA031C9C6111D178CD8B
1,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,,,,,,TUBULES,ADULT,,,,F,audax,,30195,,,,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-8-TUBULES_S82,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403797,8E4579F790957232C741689D7A3B118D
2,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,,,,,,FEMUR,ADULT,,,,F,audax,,30195,,,,,,FCONT.7_FEMUR,SAMN43369243,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-7-LEG_S50,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403799,DF916039886A02DE667F0AC833CE3896
3,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,,,,,,TUBULES,ADULT,,,,F,audax,,30195,,,,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-6-TUBULES_S80,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403800,A1C8B445AFB9B71E688D39363B2D0CDA
4,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,,,,,,TUBULES,ADULT,,,,F,audax,,30195,,,,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-TUBULES_S79,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403801,25096965DC7FC79E4610D58310A86784
5,SRX25830020,SRP489847,Illumina HiSeq 1500,SRS22459241,,,,,,FEMUR,ADULT,,,,F,audax,,30195,,,,,,FCONT.5_FEMUR,SAMN43369242,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-LEG_S48,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403802,F741CADC47A85FF967CFD7E1D8D08AC1
6,SRX25830019,SRP489847,Illumina HiSeq 1500,SRS22459240,,,,,,TUBULES,ADULT,,,,F,audax,,30195,,,,,,FCONT.4_TUBULES,SAMN43369256,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-4-TUBULES_S78,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403803,C7F2E04FEFD8088F80BF07C124DB6C30
7,SRX25830018,SRP489847,Illumina HiSeq 1500,SRS224592

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['BRAIN' 'FEMUR' 'TUBULES']


In [11]:

# BRAIN - verified fbbt xref
library.loc[library["infoOrgan"] == "BRAIN", "anatId"] = "UBERON:0000955"
library.loc[library["infoOrgan"] == "BRAIN", "anatName"] = "brain"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "BRAIN", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "BRAIN", "anatBiologicalStatus"] = "not documented"

# FEMUR - i think this is the best match - 
# i don't think it makes sense to use uberon term femur here, it doesn't xref fbbt 
# there is no femur of insect
# parent term for femur is segment of leg, which has xref in uberon insect segment of leg
library.loc[library["infoOrgan"] == "FEMUR", "anatId"] = "UBERON:6007150"
library.loc[library["infoOrgan"] == "FEMUR", "anatName"] = "insect segment of leg"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "FEMUR", "anatAnnotationStatus"] = "missing child term"
library.loc[library["infoOrgan"] == "FEMUR", "anatBiologicalStatus"] = "not documented"

# TUBULES - Malpighian tubules - verified fbbt xref
library.loc[library["infoOrgan"] == "TUBULES", "anatId"] = "UBERON:0001054"
library.loc[library["infoOrgan"] == "TUBULES", "anatName"] = "Malpighian tubule"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "TUBULES", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "TUBULES", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.9_FEMUR,SAMN43369244,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-9-LEG_S52,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403796,C2BC35F34CB9DA031C9C6111D178CD8B
1,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-8-TUBULES_S82,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403797,8E4579F790957232C741689D7A3B118D
2,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.7_FEMUR,SAMN43369243,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-7-LEG_S50,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403799,DF916039886A02DE667F0AC833CE3896
3,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-6-TUBULES_S80,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403800,A1C8B445AFB9B71E688D39363B2D0CDA
4,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-TUBULES_S79,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403801,25096965DC7FC79E4610D58310A86784
5,SRX25830020,SRP489847,Illumina HiSeq 1500,SRS22459241,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile ad

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src)

In [6]:
unique_sorted(library, "infoStage")

['ADULT' 'adult']


In [7]:
# all - looks like we are using post-juvenile adult stage for adult insects, seems appropriate to me
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,,,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,,,perfect match,F,audax,,30195,,,,,,FCONT.9_FEMUR,SAMN43369244,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-9-LEG_S52,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403796,C2BC35F34CB9DA031C9C6111D178CD8B
1,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,,,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,,,perfect match,F,audax,,30195,,,,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-8-TUBULES_S82,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403797,8E4579F790957232C741689D7A3B118D
2,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,,,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,,,perfect match,F,audax,,30195,,,,,,FCONT.7_FEMUR,SAMN43369243,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-7-LEG_S50,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403799,DF916039886A02DE667F0AC833CE3896
3,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,,,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,,,perfect match,F,audax,,30195,,,,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-6-TUBULES_S80,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403800,A1C8B445AFB9B71E688D39363B2D0CDA
4,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,,,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,,,perfect match,F,audax,,30195,,,,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-TUBULES_S79,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403801,25096965DC7FC79E4610D58310A86784
5,SRX25830020,SRP489847,Illumina HiSeq 1500,SRS22459241,,,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,,,perfect match,F,audax,,30195,,,,,,FCONT.5_FEMUR,SAMN43369242,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-LEG_S48,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403802,F741CADC47A85FF967CFD7E1D8D08AC1
6,SRX25830019,SRP489847,Illumina HiSeq 1500,SRS22459240,,,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,,,perfect match,F,audax,,30195,,,,,,FCONT.4_TUBULES,SAMN43369256,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [12]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra II Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# library selection in SRA is polyA
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.9_FEMUR,SAMN43369244,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-9-LEG_S52,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403796,C2BC35F34CB9DA031C9C6111D178CD8B
1,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-8-TUBULES_S82,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403797,8E4579F790957232C741689D7A3B118D
2,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.7_FEMUR,SAMN43369243,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-7-LEG_S50,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403799,DF916039886A02DE667F0AC833CE3896
3,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-6-TUBULES_S80,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403800,A1C8B445AFB9B71E688D39363B2D0CDA
4,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-TUBULES_S79,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403801,25096965DC7FC79E4610D58310A86784
5,SRX25830020,SRP489847,Illumina HiSeq 1500,SRS22459241,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile ad

#### EFO
i want to add caudal term for femur since they specify hind femurs

In [15]:
library.loc[library["infoOrgan"] == "FEMUR", "EFOid"] = "EFO:0001908"
library.loc[library["infoOrgan"] == "FEMUR", "EFOname"] = "caudal"
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.9_FEMUR,SAMN43369244,,,,,EFO:0001908,caudal,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-9-LEG_S52,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403796,C2BC35F34CB9DA031C9C6111D178CD8B
1,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-8-TUBULES_S82,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403797,8E4579F790957232C741689D7A3B118D
2,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.7_FEMUR,SAMN43369243,,,,,EFO:0001908,caudal,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-7-LEG_S50,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403799,DF916039886A02DE667F0AC833CE3896
3,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-6-TUBULES_S80,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403800,A1C8B445AFB9B71E688D39363B2D0CDA
4,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,,27/08/2025,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-TUBULES_S79,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403801,25096965DC7FC79E4610D58310A86784
5,SRX25830020,SRP489847,Illumina HiSeq 1500,SRS22459241,UBERON:6007150,insect segment of l

#### globin, replicates

In [13]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [16]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2024-08-27'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.9_FEMUR,SAMN43369244,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-9-LEG_S52,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403796,C2BC35F34CB9DA031C9C6111D178CD8B
1,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,SAC,2024-08-27,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-8-TUBULES_S82,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403797,8E4579F790957232C741689D7A3B118D
2,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.7_FEMUR,SAMN43369243,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-7-LEG_S50,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403799,DF916039886A02DE667F0AC833CE3896
3,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,SAC,2024-08-27,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-6-TUBULES_S80,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403800,A1C8B445AFB9B71E688D39363B2D0CDA
4,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,SAC,2024-08-27,Workers exposed over 12 days to 4.4 ppb of control or clothianidin orally in microcolonies of 6; 3 bees sampled per microcolony; Tissues dissected in RNALater and processed in pools of three representing microcolony levels,,GC-AW-9997-FCONT-5-TUBULES_S79,,,,ADULT,,TRANSCRIPTOMIC,PolyA,SRR30403801,25096965DC7FC79E4610D58310A86784
5,SRX25830020,SRP489847,Illumina HiSeq 1500,SRS22459241,UBERON:6007150,inse

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [17]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.9_FEMUR,SAMN43369244,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27
1,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,SAC,2024-08-27
2,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.7_FEMUR,SAMN43369243,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27
3,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,SAC,2024-08-27
4,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,SAC,2024-08-27
5,SRX25830020,SRP489847,Illumina HiSeq 1500,SRS22459241,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.5_FEMUR,SAMN43369242,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27
6,SRX25830019,SRP489847,Illumina HiSeq 1500,SRS22459240,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.4_TUBULES,SAMN43369256,,,,,,,,CONTROL,,SAC,2024-08-27
7,SRX25830018,SRP489847,Illumina HiSeq 1500,SRS22459239,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.4_FEMUR,SAMN43369241,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27
8,SRX25830017,SRP489847,Illumina HiSeq 1500,SRS22459238,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.3_FEMUR,SAMN43369240,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27
9,SRX25830016,SRP489847,Illumina HiSeq 1500,SRS22459237,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.2_TUBULES7,SAMN43369255,,,,,,,,CONTROL,,SAC,2024-08-27


### experiment annotations

In [18]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP489847,Acute and chronic pesticide exposure trigger fundamentally different molecular responses in bumble bee brains,"Beneficial wild insects including pollinators encounter various pesticide exposure conditions, from brief high concentrations to continuous low-level exposure. It is critical to understand how exposure patterns influence the effects of pesticides so that we can improve environmental risk assessments and how we develop new compounds. Unfortunately, this knowledge remains limited. To clarify whether different exposure schemes disrupt the physiology of pollinators in similar manners, we exposed bumble bees to acute and chronic treatments of three different pesticides that target the same neural receptors: acetamiprid, clothianidin, and sulfoxaflor. Gene expression profiling enabled us to compare the effects of these treatments on the brain in a high-resolution manner. Surprisingly, acute and chronic exposure schemes affected largely non-overlapping sets of genes: acute exposure caused distinct regulatory changes in cellular and molecular processes, rather than amplifying the effects of prolonged low-dose exposure. In contrast, different pesticides under the same exposure scheme showed more comparable effects than the same pesticide under different exposure scheme. These findings show that the mode of exposure critically determines the effects of pesticides. Our results thus challenge the common practice of extrapolating toxicity measures from one exposure scenario and signal the need for more sensitive toxicity testing practices.",SRA,,,,,,,PRJNA1076820,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [19]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

26

In [20]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
#experiment.loc[:,'projectTags'] = '' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP489847,Acute and chronic pesticide exposure trigger fundamentally different molecular responses in bumble bee brains,"Beneficial wild insects including pollinators encounter various pesticide exposure conditions, from brief high concentrations to continuous low-level exposure. It is critical to understand how exposure patterns influence the effects of pesticides so that we can improve environmental risk assessments and how we develop new compounds. Unfortunately, this knowledge remains limited. To clarify whether different exposure schemes disrupt the physiology of pollinators in similar manners, we exposed bumble bees to acute and chronic treatments of three different pesticides that target the same neural receptors: acetamiprid, clothianidin, and sulfoxaflor. Gene expression profiling enabled us to compare the effects of these treatments on the brain in a high-resolution manner. Surprisingly, acute and chronic exposure schemes affected largely non-overlapping sets of genes: acute exposure caused distinct regulatory changes in cellular and molecular processes, rather than amplifying the effects of prolonged low-dose exposure. In contrast, different pesticides under the same exposure scheme showed more comparable effects than the same pesticide under different exposure scheme. These findings show that the mode of exposure critically determines the effects of pesticides. Our results thus challenge the common practice of extrapolating toxicity measures from one exposure scenario and signal the need for more sensitive toxicity testing practices.",SRA,partial,,26,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1076820,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [21]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '40069737'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11900027/'
experiment.loc[:,'DOI'] = '10.1186/s12915-025-02169-z'
experiment.loc[:,'xrefs'] = 'https://www.sciencedirect.com/science/article/pii/S0048969724084201[paper]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP489847,Acute and chronic pesticide exposure trigger fundamentally different molecular responses in bumble bee brains,"Beneficial wild insects including pollinators encounter various pesticide exposure conditions, from brief high concentrations to continuous low-level exposure. It is critical to understand how exposure patterns influence the effects of pesticides so that we can improve environmental risk assessments and how we develop new compounds. Unfortunately, this knowledge remains limited. To clarify whether different exposure schemes disrupt the physiology of pollinators in similar manners, we exposed bumble bees to acute and chronic treatments of three different pesticides that target the same neural receptors: acetamiprid, clothianidin, and sulfoxaflor. Gene expression profiling enabled us to compare the effects of these treatments on the brain in a high-resolution manner. Surprisingly, acute and chronic exposure schemes affected largely non-overlapping sets of genes: acute exposure caused distinct regulatory changes in cellular and molecular processes, rather than amplifying the effects of prolonged low-dose exposure. In contrast, different pesticides under the same exposure scheme showed more comparable effects than the same pesticide under different exposure scheme. These findings show that the mode of exposure critically determines the effects of pesticides. Our results thus challenge the common practice of extrapolating toxicity measures from one exposure scenario and signal the need for more sensitive toxicity testing practices.",SRA,partial,,26,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1076820,40069737,https://pmc.ncbi.nlm.nih.gov/articles/PMC11900027/,10.1186/s12915-025-02169-z,https://www.sciencedirect.com/science/article/pii/S0048969724084201[paper],


#### comments

In [22]:
experiment.loc[:,'comment'] = 'removed samples treated with pesticides, these libraries come from two different papers'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP489847,Acute and chronic pesticide exposure trigger fundamentally different molecular responses in bumble bee brains,"Beneficial wild insects including pollinators encounter various pesticide exposure conditions, from brief high concentrations to continuous low-level exposure. It is critical to understand how exposure patterns influence the effects of pesticides so that we can improve environmental risk assessments and how we develop new compounds. Unfortunately, this knowledge remains limited. To clarify whether different exposure schemes disrupt the physiology of pollinators in similar manners, we exposed bumble bees to acute and chronic treatments of three different pesticides that target the same neural receptors: acetamiprid, clothianidin, and sulfoxaflor. Gene expression profiling enabled us to compare the effects of these treatments on the brain in a high-resolution manner. Surprisingly, acute and chronic exposure schemes affected largely non-overlapping sets of genes: acute exposure caused distinct regulatory changes in cellular and molecular processes, rather than amplifying the effects of prolonged low-dose exposure. In contrast, different pesticides under the same exposure scheme showed more comparable effects than the same pesticide under different exposure scheme. These findings show that the mode of exposure critically determines the effects of pesticides. Our results thus challenge the common practice of extrapolating toxicity measures from one exposure scenario and signal the need for more sensitive toxicity testing practices.",SRA,partial,,26,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1076820,40069737,https://pmc.ncbi.nlm.nih.gov/articles/PMC11900027/,10.1186/s12915-025-02169-z,https://www.sciencedirect.com/science/article/pii/S0048969724084201[paper],"removed samples treated with pesticides, these libraries come from two different papers"


#### save complete file

In [23]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [24]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [25]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [26]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
48827,SRX157961,SRP014139,Illumina HiSeq 2000,SRS348676,UBERON:0001264,pancreas,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,pancreas,unknown,perfect match,not documented,perfect match,M,,,246437,,full_length,polyA,,,pancreas,"SAMN01085956,GSM957058",,,,,,,,,,SAC,2024-08-27
48828,SRX157960,SRP014139,Illumina HiSeq 2000,SRS348675,UBERON:0002113,kidney,UBERON:0000104,life cycle,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,kidney,unknown,perfect match,not documented,perfect match,M,,,246437,,full_length,polyA,,,kidney,"SAMN01085955,GSM957057",,,,,,,,,,SAC,2024-08-27
48829,SRX25830026,SRP489847,Illumina HiSeq 1500,SRS22459247,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.9_FEMUR,SAMN43369244,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27
48830,SRX25830025,SRP489847,Illumina HiSeq 1500,SRS22459246,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.8_TUBULES,SAMN43369259,,,,,,,,CONTROL,,SAC,2024-08-27
48831,SRX25830023,SRP489847,Illumina HiSeq 1500,SRS22459245,UBERON:6007150,insect segment of leg,UBERON:0000113,post-juvenile adult stage,,FEMUR,ADULT,missing child term,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.7_FEMUR,SAMN43369243,,,,,EFO:0001908,caudal,,CONTROL,,SAC,2024-08-27
48832,SRX25830022,SRP489847,Illumina HiSeq 1500,SRS22459243,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.6_TUBULES,SAMN43369258,,,,,,,,CONTROL,,SAC,2024-08-27
48833,SRX25830021,SRP489847,Illumina HiSeq 1500,SRS22459242,UBERON:0001054,Malpighian tubule,UBERON:0000113,post-juvenile adult stage,,TUBULES,ADULT,perfect match,not documented,perfect match,F,audax,,30195,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,FCONT.5_TUBULES,SAMN43369257,,,,,,,,CONTROL,,SAC,2024-08-27


In [27]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
915,SRP029881,Transcriptome sequencing of Fukomys damarensis,Deep sequencing of mRNA from Fukomys damarensi...,SRA,total,,16,mRNA-seq Lib Prep Kit for Illumina,full_length,GSE50726,PRJNA218853,25176646,https://pmc.ncbi.nlm.nih.gov/articles/PMC4350764/,10.1016/j.celrep.2014.07.030,,
916,SRP014139,Transcriptome sequencing of Chinese tree shrew...,Deep sequencing of mRNA from Chinese tree shre...,SRA,total,,7,,full_length,GSE39150,PRJNA170104,23385571,https://www.nature.com/articles/ncomms2416,10.1038/ncomms2416,,
917,SRP489847,Acute and chronic pesticide exposure trigger f...,Beneficial wild insects including pollinators ...,SRA,partial,,26,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1076820,40069737,https://pmc.ncbi.nlm.nih.gov/articles/PMC11900...,10.1186/s12915-025-02169-z,https://www.sciencedirect.com/science/article/...,"removed samples treated with pesticides, these..."


### add annotations to git

In [28]:
! git pull

Already up to date.


In [29]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [30]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop 9d45b65] adding annotated bulk experiment SRP489847
 2 files changed, 27 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.82 KiB | 961.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git
   3a1eaa3..9d45b65  develop -> develop


### add annotation folder and script to git

In [ ]:
! git status

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push